In [1]:
import json
from pathlib import Path
with Path("../qstn_data/conditions.json").open(encoding="utf-8") as file:
    conditions: dict = json.load(file)

with Path("../qstn_data/moderators.json").open(encoding="utf-8") as file:
    demographics: dict = json.load(file)

In [2]:
from qstn.prompt_builder import LLMPrompt
from qstn.utilities import placeholder, create_one_dataframe
from qstn.survey_manager import conduct_survey_battery
import json

SYSTEM_PROMPT = (
    "You will be given a demographic and a set of questions. Your task is to predict the mean answer of this demographic for every question.\n"
    "ONLY respond in a JSON, that includes your prediction for every question in the following format:\n"
    "{\n"
    'trust_competence_1: "Your prediction e.g., 50"\n'
    "//All Other Questions\n"
    'behavior_donate: "Your prediction e.g., 50"\n'
    "}"
)

question_placeholder = "{{{{QUESTION_PLACEHOLDER}}}}"

PROMPT = (
    "Predict the mean answer for each of the following questions for a represantative sample of 5000 people of this demographic. {demographic_name}: {demographic_value}.\n"
    "Before the participants were asked these questions, they were instructed to read the following text:\n\n"
    "{text}\n\n"
    "Here are all questions. The format is QUESTION_ID: QUESTION? ANSWER_OPTIONS\n"
    f"{question_placeholder}"
)

all_llm_prompts = []
prompt_metadata = []

for cond_key, cond_val in conditions.items():
    for dem_key, dem_val in demographics.items():
        for variant_index, cond in enumerate(cond_val):
            for dem in dem_val:
                prompt = LLMPrompt(
                    questionnaire_name=f"{cond_key}_{dem_key}_{dem}",
                    questionnaire_source="../qstn_data/questionnaire.csv",
                    system_prompt=SYSTEM_PROMPT,
                    prompt=PROMPT.format(
                        demographic_name=dem_key,
                        demographic_value=dem,
                        text=cond,
                    ),
                )
                answer_options = {}
                question_stems = []
                for question in prompt.get_questions():
                    answer_options[question.item_id] = question.answer_options
                    question_stems.append(
                        f"{question.item_id}: {placeholder.QUESTION_CONTENT} {placeholder.PROMPT_OPTIONS}"
                    )
                prompt.prepare_prompt(question_stem=question_stems, answer_options=answer_options)
                all_llm_prompts.append(prompt)
                prompt_metadata.append(
                    {
                        "condition": cond_key,
                        "condition_variant": variant_index,
                        "moderator": dem_key,
                        "moderator_level": dem,
                    }
                )

print(f"Built {len(all_llm_prompts)} prompts.")


Built 459 prompts.


In [ ]:
from vllm import LLM
model_id = "google/gemma-4-E2B-it"
model = LLM(model_id, max_model_len=5000, gpu_memory_utilization=0.85, dtype="bfloat16")

INFO 07-22 15:07:41 [api_utils.py:273] non-default args: {'dtype': 'bfloat16', 'max_model_len': 5000, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'model': 'google/gemma-4-E2B-it'}


INFO 07-22 15:07:42 [model.py:619] Resolved architecture: Gemma4ForConditionalGeneration
INFO 07-22 15:07:42 [model.py:1776] Using max model len 5000
INFO 07-22 15:07:42 [scheduler.py:252] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 07-22 15:07:42 [config.py:240] Gemma4 model has heterogeneous head dimensions (head_dim=256, global_head_dim=512). FA4 not available, forcing TRITON_ATTN backend.
INFO 07-22 15:07:42 [vllm.py:1042] Asynchronous scheduling is enabled.
INFO 07-22 15:07:42 [kernel.py:292] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
(EngineCore pid=1639429) INFO 07-22 15:08:15 [core.py:114] Initializing a V1 LLM engine (v0.25.1) with config: model='google/gemma-4-E2B-it', speculative_config=None, tokenizer='google/gemma-4-E2B-it', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=5

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


(EngineCore pid=1639429) INFO 07-22 15:08:19 [default_loader.py:430] Loading weights took 1.26 seconds
(EngineCore pid=1639429) INFO 07-22 15:08:20 [model_runner.py:302] Model loading took 9.9 GiB and 4.255257 seconds
(EngineCore pid=1639429) INFO 07-22 15:08:20 [topk_topp_sampler.py:55] Using FlashInfer for top-p & top-k sampling.
(EngineCore pid=1639429) INFO 07-22 15:08:23 [backends.py:1089] Using cache directory: /home/maxi/.cache/vllm/torch_compile_cache/b8557c446d/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=1639429) INFO 07-22 15:08:23 [backends.py:1148] Dynamo bytecode transform time: 1.79 s
(EngineCore pid=1639429) INFO 07-22 15:08:24 [backends.py:292] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 1.559 s
(EngineCore pid=1639429) INFO 07-22 15:08:24 [decorators.py:311] Directly load AOT compilation from path /home/maxi/.cache/vllm/torch_compile_cache/torch_aot_compile/41feaf0372771218fa36493a7ca7cc9158c208cde4536e856abe264db

Capturing CUDA graphs (FULL): 100%|██████████| 35/35 [00:01<00:00, 32.60it/s]


(EngineCore pid=1639429) INFO 07-22 15:08:31 [model_runner.py:722] Graph capturing finished in 4 secs, took 0.67 GiB
(EngineCore pid=1639429) INFO 07-22 15:08:31 [jit_monitor.py:73] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.
(EngineCore pid=1639429) INFO 07-22 15:08:32 [core.py:337] init engine (profile, create kv cache, warmup model) took 11.47 s (compilation: 3.59 s)
(EngineCore pid=1639429) INFO 07-22 15:08:32 [kernel.py:292] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])


(EngineCore pid=1639429) WARNING 07-22 15:08:39 [jit_monitor.py:129] Triton kernel JIT compilation during inference: kernel_unified_attention. This causes a latency spike; consider extending warmup to cover this shape/config.


In [4]:
survey_result = conduct_survey_battery(model, all_llm_prompts, max_tokens=5000, chat_template_kwargs={"enable_thinking": True}, reasoning_start_token="<|channel>thought", reasoning_end_token="<channel|>")

Running survey steps:   0%|          | 0/1 [00:00<?, ?it/s]

Rendering conversations:   0%|          | 0/459 [00:00<?, ?it/s]

INFO 07-22 15:08:39 [hf.py:548] Detected the chat template content format to be 'openai'. You can set `--chat-template-content-format` to override this.


Processed prompts: 100%|██████████| 459/459 [06:50<00:00,  1.12it/s, est. speed input: 2629.40 toks/s, output: 1742.82 toks/s]


In [5]:
# Parse and export the completed survey results.
#
# This cell expects survey_result and prompt_metadata from the earlier cells.
# It preserves the model's full response and reasoning, then writes parsed
# records and the benchmark-format prediction CSV.

import json
import re
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from json_repair import repair_json

RUN_BASENAME = re.sub(r"[^A-Za-z0-9._-]+", "_", model_id).strip("_")

PROJECT_ROOT = Path.cwd().parent
if not (PROJECT_ROOT / "qstn_data").is_dir():
    PROJECT_ROOT = Path.cwd()

RAW_RESULTS_DIR = PROJECT_ROOT / "raw_results"
RESULTS_DIR = PROJECT_ROOT / "results"
PREDICTIONS_DIR = PROJECT_ROOT / "predictions"
for directory in (RAW_RESULTS_DIR, RESULTS_DIR, PREDICTIONS_DIR):
    directory.mkdir(exist_ok=True)


def parse_llm_json(response: str) -> tuple[dict | None, str | None]:
    """Parse a model response, using json_repair only after strict JSON fails."""
    try:
        value = json.loads(response)
    except json.JSONDecodeError:
        try:
            value = repair_json(response, return_objects=True)
        except Exception as exc:
            return None, f"{type(exc).__name__}: {exc}"

    if not isinstance(value, dict):
        return None, f"Expected a JSON object, got {type(value).__name__}"
    return value, None


def as_number(value) -> float | None:
    """Extract a numeric prediction, including values such as 'about 52'."""
    if isinstance(value, bool):
        return float(value)
    if isinstance(value, (int, float)):
        return float(value)
    if isinstance(value, str):
        match = re.search(r"-?\d+(?:\.\d+)?", value.replace(",", ""))
        if match:
            return float(match.group())
    return None


def newsletter_as_number(value) -> float | None:
    if isinstance(value, str):
        normalized = value.strip().casefold()
        if normalized in {"yes", "true", "1"}:
            return 1.0
        if normalized in {"no", "false", "0"}:
            return 0.0
    return as_number(value)


def mean_of(answers: dict, keys: list[str]) -> float | None:
    values = [as_number(answers.get(key)) for key in keys]
    return sum(values) / len(values) if all(value is not None for value in values) else None


def outcome_values(answers: dict) -> dict[str, float]:
    """Map item-level answers into the scored benchmark outcomes."""
    values = {
        "trust_multidimensional": mean_of(
            answers,
            [
                "trust_competence_1", "trust_competence_2", "trust_competence_3",
                "trust_integrity_1", "trust_integrity_2", "trust_integrity_3",
                "trust_benevolence_1", "trust_benevolence_2", "trust_benevolence_3",
                "trust_openness_1", "trust_openness_2", "trust_openness_3",
            ],
        ),
        "trust_post": as_number(answers.get("trust_post")),
        "distrust_post": as_number(answers.get("distrust_post")),
        "donation_ams": as_number(answers.get("donation_ams")),
        "newsletter_signup": newsletter_as_number(answers.get("newsletter_signup")),
        "funding_perceptions": (
            100 - as_number(answers["funding_perceptions"])
            if as_number(answers.get("funding_perceptions")) is not None
            else None
        ),
        "inst_trust_mean": mean_of(
            answers,
            [
                "inst_trust_epa", "inst_trust_nasa", "inst_trust_noaa",
                "inst_trust_universities", "inst_trust_federal_gov",
            ],
        ),
        "policy_role_mean": mean_of(
            answers, ["policy_role_1", "policy_role_2", "policy_role_3", "policy_role_4"]
        ),
        "belief_post": as_number(answers.get("belief_post")),
        "concern_mean": mean_of(answers, ["concern_1", "concern_2", "concern_3"]),
        "behavior_mean": mean_of(
            answers,
            [
                "behavior_meat", "behavior_transport", "behavior_solar",
                "behavior_fly", "behavior_talk", "behavior_donate",
            ],
        ),
        "policy_general": as_number(answers.get("policy_general")),
        "policy_specific_mean": mean_of(
            answers, [f"policy_specific_{index}" for index in range(1, 8)]
        ),
    }
    return {outcome: value for outcome, value in values.items() if value is not None}


raw_records = []
parsed_records = []
for metadata, inference_result in zip(prompt_metadata, survey_result, strict=True):
    for item_id, response in inference_result.results.items():
        parsed_json, parse_error = parse_llm_json(response.llm_response)
        raw_records.append(
            {
                **metadata,
                "questionnaire_name": inference_result.questionnaire.questionnaire_name,
                "item_id": item_id,
                "question": response.question,
                "reasoning": response.reasoning,
                "llm_response": response.llm_response,
                "logprobs": response.logprobs,
                "parsed_with_repair": parse_error is None and not response.llm_response.lstrip().startswith("{"),
                "parse_error": parse_error,
            }
        )
        if parsed_json is not None:
            parsed_records.append({**metadata, "answers": parsed_json})

timestamp = datetime.now(timezone.utc).isoformat()
raw_payload = {
    "model": model_id,
    "created_at": timestamp,
    "records": raw_records,
}
parsed_payload = {
    "model": model_id,
    "created_at": timestamp,
    "records": parsed_records,
}

raw_path = RAW_RESULTS_DIR / f"{RUN_BASENAME}_T2_primary_raw.json"
results_path = RESULTS_DIR / f"{RUN_BASENAME}_T2_primary_parsed.json"
main_predictions_path = PREDICTIONS_DIR / f"{RUN_BASENAME}_T2_primary_main.csv"
moderator_predictions_path = PREDICTIONS_DIR / f"{RUN_BASENAME}_T2_primary_moderator.csv"
raw_path.write_text(json.dumps(raw_payload, indent=2, ensure_ascii=False, default=str) + "\n", encoding="utf-8")
results_path.write_text(json.dumps(parsed_payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")

prediction_rows = []
for record in parsed_records:
    for outcome, mean in outcome_values(record["answers"]).items():
        prediction_rows.append({**{key: record[key] for key in ("condition", "moderator", "moderator_level")}, "outcome": outcome, "mean": mean})

prediction_values = pd.DataFrame(prediction_rows)

# T2 main: one mean per condition and outcome. Because this notebook runs every
# moderator level, this is their unweighted mean (and also averages text variants).
main_predictions = pd.DataFrame(columns=["condition", "outcome", "mean"])
moderator_predictions = pd.DataFrame(
    columns=["condition", "moderator", "moderator_level", "outcome", "mean"]
)
if not prediction_values.empty:
    main_predictions = (
        prediction_values
        .groupby(["condition", "outcome"], as_index=False)["mean"]
        .mean()
        .sort_values(["condition", "outcome"])
    )
    moderator_predictions = (
        prediction_values
        .groupby(["condition", "moderator", "moderator_level", "outcome"], as_index=False)["mean"]
        .mean()
        .sort_values(["condition", "moderator", "moderator_level", "outcome"])
    )

main_predictions.to_csv(main_predictions_path, index=False)
moderator_predictions.to_csv(moderator_predictions_path, index=False)

print(f"Raw responses: {raw_path}")
print(f"Parsed responses: {results_path}")
print(f"Main predictions: {main_predictions_path}")
print(f"Moderator predictions: {moderator_predictions_path}")
print(f"Parsed {len(parsed_records)} of {len(raw_records)} response records.")


Raw responses: /home/maxi/Documents/2026/silicon_sampling_benchmark/raw_results/google_gemma-4-E2B-it_T2_primary_raw.json
Parsed responses: /home/maxi/Documents/2026/silicon_sampling_benchmark/results/google_gemma-4-E2B-it_T2_primary_parsed.json
Main predictions: /home/maxi/Documents/2026/silicon_sampling_benchmark/predictions/google_gemma-4-E2B-it_T2_primary_main.csv
Moderator predictions: /home/maxi/Documents/2026/silicon_sampling_benchmark/predictions/google_gemma-4-E2B-it_T2_primary_moderator.csv
Parsed 458 of 459 response records.
